# Tutorial 11: Particle Swarm Optimization

**Prerequisites:** T10 (Genetic Algorithms)  
**Up Next:** T12 — Differential Evolution

---

## Concept

**Particle Swarm Optimization (PSO)** models a swarm of particles flying through the search space. Each particle has a position and velocity. It is attracted toward:

- Its **personal best** position (the best location it has visited so far).
- The **global best** position (the best location any particle has visited).

An **inertia weight** $w$ controls how much a particle keeps its current velocity vs. being pulled toward the bests. This creates a balance between **exploration** (flying freely) and **exploitation** (converging on known good regions).

## Key Takeaway

> **PSO balances exploration (inertia) and exploitation (attraction to personal and global bests). The swarm collectively discovers good regions without requiring gradient information.**

## Math Core

Each particle $i$ has position $\mathbf{x}_i$ and velocity $\mathbf{v}_i$. At each iteration:

$$\mathbf{v}_i \leftarrow w\,\mathbf{v}_i + c_1 r_1 (\mathbf{p}_i - \mathbf{x}_i) + c_2 r_2 (\mathbf{g} - \mathbf{x}_i)$$

$$\mathbf{x}_i \leftarrow \mathbf{x}_i + \mathbf{v}_i$$

where:
- $w$ = inertia weight (typically 0.4--0.9)
- $c_1$ = cognitive coefficient (attraction to personal best $\mathbf{p}_i$)
- $c_2$ = social coefficient (attraction to global best $\mathbf{g}$)
- $r_1, r_2 \sim U(0,1)$ are random vectors

## Code

In [ ]:
from dms.mechanisms.fourbar import FourBar
from dms.curves import CompareCurves
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

L0, L1 = 1.0, 1.0
PX, PY = 0.5, 0.3
TARGETS = np.array([
    [1.0, 1.5], [0.75, 1.5], [0.5, 1.5],
    [0.25, 1.5], [0.0, 1.5], [-0.25, 1.5]
])
THETAS = np.linspace(np.deg2rad(60), np.deg2rad(120), len(TARGETS))

In [ ]:
def fourbar_fk(theta1, l0, l1, l2, l3):
    ax, ay = l1 * np.cos(theta1), l1 * np.sin(theta1)
    dx, dy = ax - l0, ay
    d = np.sqrt(dx**2 + dy**2)
    cos_alpha = (l3**2 + d**2 - l2**2) / (2 * l3 * d)
    if np.abs(cos_alpha) > 1:
        return None
    alpha = np.arccos(np.clip(cos_alpha, -1, 1))
    beta = np.arctan2(dy, dx)
    theta3 = beta + alpha
    bx = l0 + l3 * np.cos(theta3)
    by = l3 * np.sin(theta3)
    theta2 = np.arctan2(by - ay, bx - ax)
    return theta2, theta3


def coupler_point(theta1, l2, l3):
    sol = fourbar_fk(theta1, L0, L1, l2, l3)
    if sol is None:
        return None
    theta2, _ = sol
    ax, ay = L1 * np.cos(theta1), L1 * np.sin(theta1)
    ux, uy = np.cos(theta2), np.sin(theta2)
    return np.array([ax + PX * ux - PY * uy,
                     ay + PX * uy + PY * ux])


def objective(x):
    l2, l3 = x
    path = []
    for theta1 in THETAS:
        pt = coupler_point(theta1, l2, l3)
        if pt is None:
            return 1e3
        path.append(pt)
    path = np.array(path)
    return CompareCurves(path[:, 0], path[:, 1],
                         TARGETS[:, 0], TARGETS[:, 1])

### Objective landscape

In [ ]:
l2_vals = np.linspace(0.3, 2.5, 40)
l3_vals = np.linspace(0.3, 2.5, 40)
L2g, L3g = np.meshgrid(l2_vals, l3_vals)
Z = np.vectorize(lambda a, b: objective([a, b]))(L2g, L3g)

### Implementing PSO from scratch

In [ ]:
def run_pso(objective, bounds, n_particles=40, n_iter=80,
            w=0.7, c1=1.5, c2=1.5, seed=42):
    """Particle Swarm Optimization."""
    rng = np.random.default_rng(seed)
    n_dim = len(bounds)
    lo = np.array([b[0] for b in bounds])
    hi = np.array([b[1] for b in bounds])
    span = hi - lo

    # Initialize positions and velocities
    pos = rng.uniform(lo, hi, size=(n_particles, n_dim))
    vel = rng.uniform(-0.2 * span, 0.2 * span, size=(n_particles, n_dim))

    # Evaluate initial fitness
    fit = np.array([objective(p) for p in pos])

    # Personal bests
    pbest_pos = pos.copy()
    pbest_fit = fit.copy()

    # Global best
    gbest_idx = np.argmin(fit)
    gbest_pos = pos[gbest_idx].copy()
    gbest_fit = fit[gbest_idx]

    best_history = [gbest_fit]
    pos_history = [pos.copy()]

    for it in range(n_iter):
        r1 = rng.random(size=(n_particles, n_dim))
        r2 = rng.random(size=(n_particles, n_dim))

        # Update velocities
        vel = (w * vel
               + c1 * r1 * (pbest_pos - pos)
               + c2 * r2 * (gbest_pos - pos))

        # Update positions
        pos = pos + vel
        pos = np.clip(pos, lo, hi)

        # Evaluate
        fit = np.array([objective(p) for p in pos])

        # Update personal bests
        improved = fit < pbest_fit
        pbest_pos[improved] = pos[improved]
        pbest_fit[improved] = fit[improved]

        # Update global best
        gen_best_idx = np.argmin(pbest_fit)
        if pbest_fit[gen_best_idx] < gbest_fit:
            gbest_pos = pbest_pos[gen_best_idx].copy()
            gbest_fit = pbest_fit[gen_best_idx]

        best_history.append(gbest_fit)
        pos_history.append(pos.copy())

    return gbest_pos, gbest_fit, best_history, pos_history

In [ ]:
bounds = [(0.3, 2.5), (0.3, 2.5)]
best_x, best_f, history, pos_history = run_pso(objective, bounds)

print(f'PSO best solution: l2={best_x[0]:.4f}, l3={best_x[1]:.4f}')
print(f'PSO best objective: {best_f:.4f}')

### Convergence plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history, 'b-', lw=1.5)
ax.set_xlabel('Iteration')
ax.set_ylabel('Global best objective')
ax.set_title('PSO convergence')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()

### Swarm movement on the contour plot

Snapshots of particle positions at iterations 0, 20, 50, and the final iteration.

In [ ]:
snap_iters = [0, 20, 50, len(pos_history) - 1]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, it in zip(axes, snap_iters):
    ax.contourf(L2g, L3g, np.log1p(Z), levels=30, cmap='viridis', alpha=0.8)
    swarm = pos_history[it]
    ax.scatter(swarm[:, 0], swarm[:, 1], c='red', s=15, edgecolors='white', linewidths=0.5)
    ax.set_title(f'Iter {it}')
    ax.set_xlabel(r'$\ell_2$')
    ax.set_ylabel(r'$\ell_3$')
    ax.set_xlim(0.3, 2.5)
    ax.set_ylim(0.3, 2.5)

fig.suptitle('PSO swarm snapshots', fontsize=14)
plt.tight_layout()

### Compare PSO to GA

Let's run the GA from Tutorial 10 with the same budget (40 individuals, 80 generations) and compare.

In [ ]:
def run_ga(objective, bounds, pop_size=40, n_gen=80,
           tournament_k=3, crossover_alpha=0.5,
           mutation_rate=0.3, mutation_sigma=0.15, seed=42):
    """Simple real-coded GA (from Tutorial 10)."""
    rng = np.random.default_rng(seed)
    n_dim = len(bounds)
    lo = np.array([b[0] for b in bounds])
    hi = np.array([b[1] for b in bounds])
    pop = rng.uniform(lo, hi, size=(pop_size, n_dim))
    fitness = np.array([objective(ind) for ind in pop])
    best_history = []
    for gen in range(n_gen):
        new_pop = []
        best_idx = np.argmin(fitness)
        new_pop.append(pop[best_idx].copy())
        while len(new_pop) < pop_size:
            parents = []
            for _ in range(2):
                candidates = rng.integers(0, pop_size, size=tournament_k)
                winner = candidates[np.argmin(fitness[candidates])]
                parents.append(pop[winner])
            p1, p2 = parents
            beta = rng.uniform(-crossover_alpha, 1 + crossover_alpha, size=n_dim)
            child = p1 + beta * (p2 - p1)
            for j in range(n_dim):
                if rng.random() < mutation_rate:
                    child[j] += rng.normal(0, mutation_sigma)
            child = np.clip(child, lo, hi)
            new_pop.append(child)
        pop = np.array(new_pop[:pop_size])
        fitness = np.array([objective(ind) for ind in pop])
        best_history.append(fitness.min())
    best_idx = np.argmin(fitness)
    return pop[best_idx], fitness[best_idx], best_history

ga_best_x, ga_best_f, ga_history = run_ga(objective, bounds)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ga_history, 'r-', lw=1.5, label=f'GA (best={ga_best_f:.4f})')
ax.plot(history, 'b-', lw=1.5, label=f'PSO (best={best_f:.4f})')
ax.set_xlabel('Generation / Iteration')
ax.set_ylabel('Best objective')
ax.set_title('PSO vs GA convergence')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

### Visualizing the PSO result

In [ ]:
path_pso = np.array([coupler_point(t, best_x[0], best_x[1]) for t in THETAS])

mechanism = FourBar(L0, L1, best_x[0], best_x[1])
fig, ax = plt.subplots(figsize=(6, 5))
mechanism.plot(np.deg2rad(90), ax=ax)
ax.plot(TARGETS[:, 0], TARGETS[:, 1], 'r*', ms=10, label='targets')
ax.plot(path_pso[:, 0], path_pso[:, 1], 'b.-', label='coupler path')
ax.set_aspect('equal')
ax.legend()
ax.set_title(f'PSO result: $\\ell_2$={best_x[0]:.3f}, $\\ell_3$={best_x[1]:.3f}, f={best_f:.4f}')
plt.tight_layout()

---

## Exercises

1. **Inertia weight:** Try $w = 0.3$ (more exploitation) vs $w = 0.95$ (more exploration). How does convergence change?

2. **Cognitive vs social:** Set $c_1 = 2.5, c_2 = 0.5$ (individualist swarm) then $c_1 = 0.5, c_2 = 2.5$ (conformist swarm). Which converges faster? Which finds a better solution?

3. **Linear inertia decay:** Modify the PSO to linearly decrease $w$ from 0.9 to 0.4 over the iterations. Does this improve results?

4. **Velocity clamping:** Add a maximum velocity $v_{\max} = 0.5$. Clip velocities each iteration. Does this help or hurt?